# Task 3: find the ball when the tracking loses it

**Owner:** ____

**The problem.** The tracking data loses the ball on 41% of the frames that are scored. Without a
ball we can't tell which team has it. Most of these gaps are long, not brief flickers:

| gap length | gaps | share of missing frames |
|---|---|---|
| ≤ 1 s | 17 | 0.12% |
| ≤ 5 s | 18 | 0.30% |
| ≤ 10 s | 71 | 14.8% |
| **> 10 s** | **79** | **85.2%** |

So joining up short gaps with straight lines recovers almost nothing. Instead, this notebook finds the
ball in the match video itself.

**Not covered here:** whether the ball is in play or dead (Task 4), and turning the evidence into
final labels (Task 1).


In [ ]:
import sys; sys.path.insert(0, '..')
from pipeline.common import *
from pipeline import ball_filled as bf
import matplotlib.pyplot as plt
import pandas as pd

trk = load_tracking()          # ball, ball_speed, players, player_speed, team -- all in metres
gt = load_public_gt()          # public ground truth, X outside the 5-minute window
graded = graded_mask()         # frames the scorer grades
half = half_of_frame()         # 1, 2, or 0 outside the graded halves
PUBLIC = (11185, 18684)        # the 5 minutes we have ground truth for (inclusive)


## Step 1: a ready-made ball detector

**Model.** WASB, a small neural network built to find balls in sports video. It looks at 3
consecutive frames and marks where the ball is. Tarashima, Haq, Wang, Tagawa, *Widely Applicable
Strong Baseline for Sports Ball Detection and Tracking*, BMVC 2023
([arXiv 2311.05237](https://arxiv.org/abs/2311.05237), weights from
[nttcom/WASB-SBDT](https://github.com/nttcom/WASB-SBDT)).

**Trained on.** ISSIA-CNR Soccer, a different match filmed by six fixed cameras. D'Orazio, Leo, Mosca,
Spagnolo, Mazzeo, *A semi-automatic system for ground truth generation of soccer video sequences*,
AVSS 2009. We used the published weights as they are.

**Making it fit our video.** Our ball is tiny (about 6 px), so each frame is cut into 16 overlapping
tiles of 640×360 and each tile is searched separately. We also used AdaBN: the network's internal
brightness and contrast statistics are recalculated on 300 clips of our video. This needs no labels.
Li, Wang, Shi, Liu, Hou, *Revisiting Batch Normalization For Practical Domain Adaptation*, ICLR
Workshop 2017. Detections in the crowd are thrown away, and we keep the strongest one on the pitch.

**How we score it.** On frames where the tracking does have the ball, we map its position onto the
video. A detection counts as a hit if it is within 40 px of that point.

**Results** on 7 clips of 15 s (1,152 frames with a tracking ball, 1,473 without):

| setting | ball found (frames with tracking ball) | detections that are correct | frames without tracking ball that get a detection |
|---|---|---|---|
| 960×540 tiles | 27% | 79% | 7% |
| 480×270 tiles | 20% | 28% | 54% |
| 640×360 tiles | 37% | 64% | 40% |
| **640×360 + AdaBN** | **50%** | **83%** | **17%** |

640×360 tiles with AdaBN worked best. On frames without a tracking ball there is nothing to check
against. By eye, most detections there sat at a player's feet: often the ball, sometimes a white boot.


### Training trick: hiding the ball behind players

The tracking only gives us the ball's position when the ball is clearly visible. But the frames we
care about are the hard ones, where players crowd around or cover the ball. To train for those, we cut
a player out of another part of the video and paste them over the ball. The model must still answer
with the ball's true position.

The cell below makes one example. Two details made it work:
- The ball's position is snapped to the nearest small, round, white spot. Snapping to the brightest
  pixel picked white boots.
- The player is cut out by how different their colours are from the grass around them. Cutting by
  colour hue lost heads and legs.


In [ ]:
# Synthetic occlusion: one training example, before and after pasting a player over the ball.
# Illustration only. Needs a working opencv (pip install opencv-python). Clear outputs before committing.
import cv2
from games import require
from starter.load_homography import load as load_homography, project

H_all, _, _ = load_homography()
video = cv2.VideoCapture(require(["video"])[0])
ball_n = trk["ball"] / PITCH                      # normalised 0-1, what the homography expects
players_n = trk["players"] / PITCH


def read_frame(f):
    video.set(cv2.CAP_PROP_POS_FRAMES, f)
    return video.read()[1]


def px_per_metre(f, xy_n):
    """Image pixels per metre of pitch, horizontally, at a normalised pitch position."""
    a = np.array(project(H_all[f], *xy_n))
    b = np.array(project(H_all[f], xy_n[0] + 1 / PITCH[0], xy_n[1]))
    return np.hypot(*(b - a))


def refine_ball(img, u, v, radius=40):
    """Snap the projected tracking ball (off by 5-35 px) to the nearest small, round, white blob.
    Brightest-pixel snapping picks white boots, so blobs are filtered on size and shape. None if no blob."""
    x0, y0 = int(u) - radius, int(v) - radius
    hsv = cv2.cvtColor(img[y0:y0 + 2 * radius, x0:x0 + 2 * radius], cv2.COLOR_BGR2HSV)
    white = cv2.inRange(hsv, (0, 0, 170), (180, 70, 255))
    n, _, stats, cent = cv2.connectedComponentsWithStats(white)
    best, best_d = None, np.inf
    for k in range(1, n):
        x, y, w, h, area = stats[k]
        round_enough = 0.6 < w / h < 1.6 and area > 0.55 * w * h
        if 12 <= area <= 80 and round_enough:
            d = np.hypot(cent[k][0] - radius, cent[k][1] - radius)
            if d < best_d:
                best, best_d = (x0 + int(cent[k][0]), y0 + int(cent[k][1])), d
    return best


def cut_player(img, feet, ppm, threshold=12):
    """Cut a player out around their projected feet. A pixel is "player" when its colour is far from the
    local grass (median of the window's border, in Lab, brightness down-weighted so mowing stripes stay
    grass). Hue keying loses the head and legs at this size. The window is generous because the
    projection is off by up to ~35 px; the cut-out is trimmed so its bottom is the feet. Returns (crop, alpha)."""
    w, h = int(3.0 * ppm), int(4.0 * ppm)
    x0, y0 = int(feet[0]) - w // 2, int(feet[1]) - h + int(1.2 * ppm)
    crop = img[y0:y0 + h, x0:x0 + w].copy()
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(float)
    border = np.concatenate([lab[:3].reshape(-1, 3), lab[-3:].reshape(-1, 3),
                             lab[:, :3].reshape(-1, 3), lab[:, -3:].reshape(-1, 3)])
    grass = np.median(border, axis=0)
    dist = np.sqrt((0.4 * (lab[..., 0] - grass[0])) ** 2 + ((lab[..., 1:] - grass[1:]) ** 2).sum(-1))
    body = cv2.morphologyEx((dist > threshold).astype(np.uint8), cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    n, lab_ids, stats, cent = cv2.connectedComponentsWithStats(body)
    # The player: of the sizeable blobs, the one nearest the expected feet position.
    feet_xy = np.array([w / 2, h - 1.2 * ppm])
    big = [k for k in range(1, n) if stats[k, cv2.CC_STAT_AREA] > 0.3 * ppm ** 2] or list(range(1, n))
    k = min(big, key=lambda k: np.hypot(*(cent[k] - feet_xy)))
    x, y, bw, bh, _ = stats[k]
    alpha = cv2.GaussianBlur((lab_ids == k).astype(np.float32), (3, 3), 0)
    return crop[y:y + bh, x:x + bw], alpha[y:y + bh, x:x + bw]


def paste(img, crop, alpha, feet_xy):
    """Paste crop so that its bottom-centre sits at feet_xy."""
    out = img.copy()
    h, w = alpha.shape
    x0, y0 = int(feet_xy[0]) - w // 2, int(feet_xy[1]) - h
    region = out[y0:y0 + h, x0:x0 + w].astype(float)
    out[y0:y0 + h, x0:x0 + w] = (alpha[..., None] * crop + (1 - alpha[..., None]) * region).astype(np.uint8)
    return out


# A visible, slow, uncontested ball, so the "before" shows the ball clearly.
with np.errstate(invalid="ignore"):
    dist = np.nanmin(np.linalg.norm(trk["players"] - trk["ball"][:, None], axis=2), axis=1)
ok = ~np.isnan(trk["ball"][:, 0]) & (trk["ball_speed"] < 3) & (dist > 2) & (dist < 4) & graded
for f in np.flatnonzero(ok)[len(np.flatnonzero(ok)) // 3:]:
    img = read_frame(int(f))
    snapped = refine_ball(img, *project(H_all[f], *ball_n[f]))
    if snapped is not None:
        break
f = int(f); bu, bv = snapped

# Donor: a player at least 10 m from the ball and 3 m from everyone else, cut from the same frame.
p = trk["players"][f]
far = np.linalg.norm(p - trk["ball"][f], axis=1) > 10
alone = np.sort(np.linalg.norm(p[:, None] - p[None], axis=2), axis=1)[:, 1] > 3
donor = int(np.flatnonzero(far & alone & ~np.isnan(p[:, 0]))[0])
ppm = px_per_metre(f, players_n[f, donor])
crop, alpha = cut_player(img, project(H_all[f], *players_n[f, donor]), ppm)

# Feet just in front of the ball, so the body covers it as in a contest. The label stays at (bu, bv).
occluded = paste(img, crop, alpha, (bu + 0.1 * ppm, bv + 0.25 * ppm))

zoom = lambda im: cv2.cvtColor(im[bv - 60:bv + 60, bu - 60:bu + 60], cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, im, title in zip(axes, (img, occluded), ("original (label = ring)", "player pasted over the ball")):
    ax.imshow(zoom(im)); ax.add_patch(plt.Circle((60, 60), 9, fill=False, color="red")); ax.set_title(title); ax.axis("off")
fig.suptitle(f"frame {f}: training label kept at the ball, image occluded")
plt.show()

In [ ]:
# Load weights: the released WASB soccer checkpoint, its AdaBN version (fold 4 held out), and the
# fine-tuned one. Needs the video environment (.venv-video, requirements-video.txt) as the kernel.
import os
import torch
from pipeline import ball_video as bv, ball_video_data as bvd, ball_video_labels as bvl, wasb
from pipeline import ball_video_train as bvt, occlusion as occ

device = wasb.pick_device()                                    # "mps" on the M1, else "cpu"
wasb.fetch_pretrained(wasb.PRETRAINED_PATH)                    # downloads once (Google Drive, ~6 MB)
bv.make_adabn(exclude_fold=bvd.HELD_OUT, out_path=bv.ADABN_HELDOUT_PATH)   # skips if cached (~10 min)

pretrained = wasb.load_model(wasb.PRETRAINED_PATH, device)
adabn = wasb.load_model(bv.ADABN_HELDOUT_PATH, device)
finetuned = wasb.load_model(bv.FINETUNED_WEIGHTS, device)            # committed: weights/ball_video_finetuned.pt
print("parameters:", sum(p.numel() for p in finetuned.parameters()))


## Step 2: training the detector on our own match

**Labels.** Every 5th frame where the tracking has the ball, its position snapped onto the video ball
(18,843 frames). No one labelled anything by hand.

**Training.** We started from Step 1's detector and trained it for 3 rounds on 16,179 tiles: 10,000
with the ball, 5,000 without, and 1,179 where Step 1 had wrongly seen a ball. On 40% of the ball tiles,
a player was pasted over the ball (see above). One fifth of the match was kept out of training
entirely, to test on.

**Result** on 10 clips of 15 s from the held-out fifth (1,981 frames with a tracking ball, 1,769
without):

| | Step 1 | after training |
|---|---|---|
| ball found, all frames | 33% | 56% |
| ball found, crowded (2+ players within 2 m) | 40% | 68% |
| ball found, near a goal | 52% | 81% |
| ball found, moving fast (over 15 m/s) | 19% | 26% |
| detections that are correct | 58% | 68% |
| frames without tracking ball that get a detection | 20% | 29% |

Training helped everywhere, most of all in crowded moments. A fast-moving ball is still rarely found. These clips are harder than Step 1's:
the same Step 1 model got 83% of its detections right there, but only 58% here.


In [ ]:
# Fine-tune the detector yourself. Off by default: the notebook uses the committed weights
# (weights/ball_video_finetuned.pt). Set RETRAIN = True to rebuild everything from the video and
# tracking. From scratch it takes about 7 h on an 8 GB M1 (labels 1 h, donors 5 min, tiles 2 h,
# training 4 h); each step is cached under pipeline/cache/ball_video
# For a run that outlives the notebook, use the terminal instead:
#   caffeinate -dimsu .venv-video/bin/python -c "from pipeline import ball_video_train as t; t.train()"
import pandas as pd

RETRAIN = False

if RETRAIN:
    # 1. labels: projected tracking ball snapped to a small round white blob, every 5th visible graded frame
    labels = pd.read_parquet(bvl.LABELS_PATH) if os.path.exists(bvl.LABELS_PATH) else bvl.build_labels()
    print(f"{len(labels):,} labels, snapped {labels['snapped'].mean():.0%}, held-out fold {bvd.HELD_OUT}: "
          f"{(labels['fold'] == bvd.HELD_OUT).sum():,}")

    # 2. donor players for synthetic occlusion (cut out by colour distance from the grass)
    if not os.path.exists(occ.DONORS_PATH):
        occ.save_donor_bank(occ.build_donor_bank())
    print(len(occ.load_donor_bank()), "donor players")

    # 3. tile cache: ball tiles, random ball-free tiles, and tiles centred on the AdaBN model's confident misses
    samples_path = os.path.join(bvd.CACHE_DIR, "samples.parquet")
    samples = pd.read_parquet(samples_path) if os.path.exists(samples_path) else bvd.build()
    print(samples["kind"].value_counts().to_dict())

    # 4. train: 3 epochs, batch 2 x 4 accumulation, BatchNorm frozen at the AdaBN stats; keeps the best
    #    checkpoint on whole 10-minute validation stretches (pos_hit - neg_fire). Prints every 200 steps.
    bvt.train()

    # use your weights in the cells below (the committed file is left untouched)
    finetuned = wasb.load_model(bvt.FINETUNED_PATH, device)
    print("using", bvt.FINETUNED_PATH)


In [ ]:
# Visualisation: the fine-tuned model on a held-out (fold 4) window. Top row: frames where the tracking
# has the ball (cross = tracking ball, circle = detection). Bottom row: frames where the tracking lost it.
# ~1 min on MPS. Clear outputs before committing.
window = bv.eval_windows(1, 375)[0]
n = 150
lost = window + np.flatnonzero(np.isnan(trk["ball"][window:window + 375, 0]))[0]
start = lost - n // 2                                   # centred on where the tracking loses the ball
frames = [read_frame(f) for f in range(start, start + n)]
dets = bv.detect(finetuned, frames, device)
est = np.array([bv.top_on_pitch(d, H_all[f], 0.5) for f, d in zip(range(start, start + n), dets)])
seen = ~np.isnan(trk["ball"][start:start + n, 0])
found = ~np.isnan(est[:, 0])
picks = [np.flatnonzero(m)[np.linspace(0, m.sum() - 1, min(3, m.sum())).astype(int)] for m in (seen & found, ~seen & found)]

fig, axes = plt.subplots(2, 3, figsize=(15, 6.5))
for row, idx in zip(axes, picks):
    for ax in row:
        ax.axis("off")
    for ax, i in zip(row, idx):
        f = start + i
        x, y, peak = est[i]
        ax.imshow(cv2.cvtColor(frames[i], cv2.COLOR_BGR2RGB))
        ax.add_patch(plt.Circle((x, y), 25, fill=False, color="red", lw=1.5))
        if seen[i]:
            ax.plot(*project(H_all[f], *ball_n[f]), "x", color="yellow", ms=8)
        ax.set_xlim(x - 240, x + 240); ax.set_ylim(y + 135, y - 135)       # 480x270 zoom on the detection
        ax.set_title(f"frame {f}  peak {peak:.2f}  {'tracked' if seen[i] else 'tracking lost'}", fontsize=9)
plt.tight_layout(); plt.show()
print(f"window {start}-{start + n - 1}: detection on {found[seen].mean():.0%} of tracked frames, "
      f"{found[~seen].mean():.0%} of lost-ball frames")


# Improvement on the baseline

The organisers' baseline (`starter/baseline.py`) gives the ball to the team of the nearest player. When
the tracking has no ball, it says D (dead ball). Then it smooths the labels with a moving majority vote.

Here we give the baseline our video ball wherever the tracking lost it. Both versions are scored on
the public 5 minutes with the official scorer.

**Result.** With the published smoothing window (±50 frames), the main score, spellF1, rises from
0.326 to 0.353. Per-frame accuracy (macroF1) falls from 0.621 to 0.521, because stoppages now get
labelled as one team's possession instead of D. The video ball shows who is near the ball, not whether
play has stopped, so it needs Task 4's dead-ball detection next to it.

These 5 minutes hold only about 17 possessions, so read this as a direction, not a verdict.


In [ ]:
# Baseline vs baseline + video ball, scored on the public window. Needs bv.build() output
# (bv.BALL_VIDEO_PATH, ~11.6 h on MPS, resumable). Submissions go to a temp folder, not evaluation/results.
import subprocess, tempfile
sys.path.insert(0, os.path.join("..", "starter"))
import baseline as bl

codes = bl.build()                                            # nearest player to the tracking ball, NOBODY where missing
video_ball = np.load(bv.BALL_VIDEO_PATH)["ball_video"]
fill = np.flatnonzero((codes == bl.NOBODY) & ~np.isnan(video_ball[:, 0]))
dist = np.linalg.norm(trk["players"][fill] - video_ball[fill][:, None], axis=2)
dist = np.where(np.isnan(dist), np.inf, dist)
has_player = np.isfinite(dist).any(axis=1)
filled = codes.copy()
filled[fill[has_player]] = np.asarray(trk["team"])[np.argmin(dist[has_player], axis=1)]
print(f"ball located on {np.mean(codes != bl.NOBODY):.1%} of frames -> {np.mean(filled != bl.NOBODY):.1%} with the video ball")

out = tempfile.mkdtemp()
for w in (0, 25, 50, 125):
    name = f"smooth{w}" if w else "raw"
    bl.write_submission(os.path.join(out, "results", f"baseline_{name}.csv"), bl.smooth(codes, w))
    bl.write_submission(os.path.join(out, "results", f"video_{name}.csv"), bl.smooth(filled, w))
subprocess.run([sys.executable, os.path.join("..", "evaluation", "run_evaluation.py"), "--gt", "public", "--no-detail",
                "--results", os.path.join(out, "results"), "--out", out], check=True, capture_output=True)
board = pd.read_csv(os.path.join(out, "scoreboard.csv"))
board["window"] = board["submission"].str.split("_").str[1]
board["fill"] = board["submission"].str.split("_").str[0]
display(board.pivot(index="window", columns="fill", values=["spell_f1_mean", "macro_f1", "f1_D", "poss"])
        .reindex(["raw", "smooth25", "smooth50", "smooth125"]).round(3))
